# Линейная регрессия

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler, FunctionTransformer, PolynomialFeatures
from sklearn.impute import SimpleImputer


def cv(model, X, y):
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_percentage_error', n_jobs=1)
    mape = -scores
    print(f"MAPE: {mape.mean():.6f} +- {mape.std():.6f}")
    return mape.mean()


def save_submission(model, X_train, y_train, X_test, ids, path):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    os.makedirs('out', exist_ok=True)
    pd.DataFrame({'ID': ids, 'salary_mean_net': pred}).to_csv(path, index=False)
    print('saved to', path)


log_transform = FunctionTransformer(lambda a: np.log1p(np.clip(a, 0, None)), feature_names_out='one-to-one')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.feature_extraction.text import TfidfVectorizer


## Подготовка данных

In [2]:
df_train = pd.read_csv('data/train.csv')
X = df_train.iloc[:, :-1].copy()
y = df_train.iloc[:, -1].copy()

X[['unified_address_city', 'unified_address_region']] = X[['unified_address_city', 'unified_address_region']].fillna('missing')
X = X.drop(columns=['id', 'raw_description', 'employer_id', 'name', 'raw_branded_description', 'lemmaized_wo_stopwords_raw_branded_description', 'unified_address_country'], errors='ignore')

counts = X['employer_name'].value_counts()
rare_categories = counts[counts < 200].index
X['employer_name'] = X['employer_name'].replace(rare_categories, 'Other')

for c in ['lemmaized_wo_stopwords_raw_description', 'name_clean', 'key_skills_name']:
    X[c] = X[c].fillna('')

text_cols = ['lemmaized_wo_stopwords_raw_description', 'name_clean', 'key_skills_name']
cat_cols = [c for c in X.select_dtypes(include=['object']).columns if c not in text_cols]
num_cols = [c for c in X.columns if c not in cat_cols + text_cols]


## 1. Выбор скейлера

In [3]:
def build_preprocessor(scaler_name, feature_mode='none'):
    if scaler_name == 'standard':
        scaler = StandardScaler()
    elif scaler_name == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = RobustScaler()

    num_steps = [('imp', SimpleImputer(strategy='median'))]

    if feature_mode in ['log', 'log_poly']:
        num_steps.append(('log', log_transform))

    num_steps.append(('sc', scaler))

    if feature_mode in ['poly', 'log_poly']:
        num_steps.append(('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)))

    pre = ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('txt1', TfidfVectorizer(max_features=1200, min_df=3, max_df=0.9), 'lemmaized_wo_stopwords_raw_description'),
        ('txt2', TfidfVectorizer(max_features=1200, min_df=3, max_df=0.9), 'name_clean'),
        ('txt3', TfidfVectorizer(max_features=1200, min_df=3, max_df=0.9), 'key_skills_name'),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat_cols),
    ], remainder='drop')
    return pre


### Эксперимент 1.1: StandardScaler

In [4]:
pre_std = build_preprocessor('standard', 'none')
pipe_std = Pipeline([('preprocessor', pre_std), ('model', LinearRegression())])
mape_std = cv(pipe_std, X, y)


MAPE: 0.296748 +- 0.004401


### Эксперимент 1.2: MinMaxScaler

In [5]:
pre_mm = build_preprocessor('minmax', 'none')
pipe_mm = Pipeline([('preprocessor', pre_mm), ('model', LinearRegression())])
mape_mm = cv(pipe_mm, X, y)


MAPE: 0.296746 +- 0.004404


### Эксперимент 1.3: RobustScaler

In [6]:
pre_rb = build_preprocessor('robust', 'none')
pipe_rb = Pipeline([('preprocessor', pre_rb), ('model', LinearRegression())])
mape_rb = cv(pipe_rb, X, y)


MAPE: 0.296746 +- 0.004404


In [7]:
scaler_scores = {'standard': mape_std, 'minmax': mape_mm, 'robust': mape_rb}
best_scaler = min(scaler_scores, key=scaler_scores.get)
print('Best scaler:', best_scaler, 'MAPE=', scaler_scores[best_scaler])


Best scaler: minmax MAPE= 0.29674610058638534


## 2. На лучшем скейлере сравниваем: без / log / poly / log+poly

### Эксперимент 2.1: без дополнительных фич

In [8]:
pre_none = build_preprocessor(best_scaler, 'none')
pipe_none = Pipeline([('preprocessor', pre_none), ('model', LinearRegression())])
mape_none = cv(pipe_none, X, y)


MAPE: 0.296746 +- 0.004404


### Эксперимент 2.2: логарифмические фичи

In [9]:
pre_log = build_preprocessor(best_scaler, 'log')
pipe_log = Pipeline([('preprocessor', pre_log), ('model', LinearRegression())])
mape_log = cv(pipe_log, X, y)


MAPE: 0.296746 +- 0.004404


### Эксперимент 2.3: полиномиальные фичи

In [10]:
pre_poly = build_preprocessor(best_scaler, 'poly')
pipe_poly = Pipeline([('preprocessor', pre_poly), ('model', LinearRegression())])
mape_poly = cv(pipe_poly, X, y)


MAPE: 0.296746 +- 0.004418


### Эксперимент 2.4: логарифмические + полиномиальные фичи

In [11]:
pre_log_poly = build_preprocessor(best_scaler, 'log_poly')
pipe_log_poly = Pipeline([('preprocessor', pre_log_poly), ('model', LinearRegression())])
mape_log_poly = cv(pipe_log_poly, X, y)


MAPE: 0.296746 +- 0.004418


In [12]:
feature_scores = {
    'none': mape_none,
    'log': mape_log,
    'poly': mape_poly,
    'log_poly': mape_log_poly,
}
pre_map = {
    'none': pre_none,
    'log': pre_log,
    'poly': pre_poly,
    'log_poly': pre_log_poly,
}
best_feature_mode = min(feature_scores, key=feature_scores.get)
best_pre = pre_map[best_feature_mode]
print('Best feature mode:', best_feature_mode, 'MAPE=', feature_scores[best_feature_mode])


Best feature mode: poly MAPE= 0.2967459877941809


## 3. Подбор параметров на лучшем варианте из шагов 1 и 2

In [13]:
grid = GridSearchCV(
    estimator=Pipeline([('preprocessor', best_pre), ('model', Ridge())]),
    param_grid={'model__alpha':[0.1,0.5,1.0,2.0,5.0,10.0]},
    scoring='neg_mean_absolute_percentage_error',
    cv=5,
    n_jobs=1,
    verbose=1,
)

grid.fit(X, y)
print('Best params:', grid.best_params_)
print('Best CV MAPE:', -grid.best_score_)
best_model = grid.best_estimator_


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'model__alpha': 10.0}
Best CV MAPE: 0.2836915715652205


## Предикт теста и сохранение

In [15]:
df_test = pd.read_csv('data/test_x.csv')
X_test = df_test.copy()
X_test[['unified_address_city', 'unified_address_region']] = X_test[['unified_address_city', 'unified_address_region']].fillna('missing')
X_test = X_test.drop(columns=['id', 'raw_description', 'employer_id', 'name', 'raw_branded_description', 'lemmaized_wo_stopwords_raw_branded_description', 'unified_address_country'], errors='ignore')
X_test['employer_name'] = X_test['employer_name'].replace(rare_categories, 'Other')
for c in ['lemmaized_wo_stopwords_raw_description', 'name_clean', 'key_skills_name']:
    X_test[c] = X_test[c].fillna('')

save_submission(best_model, X, y, X_test, df_test['id'], 'out/linear_regression_submission.csv')


saved to out/linear_regression_submission.csv
